# TVM Prep Guide

This is the main guide for the repository. The markdown files explain how to open the devcontainer and install dependencies; this notebook explains how to use the project once the container is ready.

Run this notebook from top to bottom after these setup commands have completed in the repository root:

```bash
bash .devcontainer/scripts/install-python-deps.sh requirements.txt
bash .devcontainer/scripts/build-tvm.sh
```

The notebook compiles a pretrained ImageNet model for the local `x86_64` target, runs the compiled TVM artifacts on `examples/assets/cat.png`, and prints the top labels. After that, it shows how the same workflow extends to other frontends, target profiles, Python runtime validation, and the C++ runner.


## What TVM Is Doing Here

Apache TVM is a compiler stack for machine-learning models. In this repository the workflow has five stages:

1. Start from a model in a framework such as PyTorch, TensorFlow/Keras, ONNX, or TFLite.
2. Convert that model into TVM Relay, which is TVM's high-level graph representation.
3. Compile Relay for a target profile, such as local x86_64 Linux or Raspberry Pi AArch64.
4. Export graph-executor artifacts: graph JSON, parameter bytes, compiled library, and metadata.
5. Load those artifacts in a runtime and run inference.

The notebook uses `compilation/compile.py` for the compile/export step and `examples/python/` for runtime execution. The code stays close to the TVM APIs so it is clear what TVM is doing.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

EXAMPLES    = ROOT / 'examples' / 'python'
COMPILATION = ROOT / 'compilation'
for p in (EXAMPLES, COMPILATION):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('Python:', sys.version.split()[0])
print('Repository:', ROOT)


## Validate The Environment

A TVM setup can fail in several places: the Python frameworks may be missing, TVM may not find its compiled shared libraries, or the cross-compilers may not exist. This cell checks those pieces before any model compilation begins.

The imports run in subprocesses so noisy framework startup logs, especially TensorFlow CUDA messages, do not make the notebook look failed when the import actually succeeded.

In [ ]:
import shutil
import subprocess

from targets import TARGETS

checks = {
    'TVM':          'import tvm; print(tvm.__version__)',
    'PyTorch':      'import torch; print(torch.__version__)',
    'TorchVision':  'import torchvision; print(torchvision.__version__)',
    'TensorFlow':   'import tensorflow as tf; print(tf.__version__)',
    'ONNX':         'import onnx; print(onnx.__version__)',
    'ONNX Runtime': 'import onnxruntime; print(onnxruntime.__version__)',
}

for label, code in checks.items():
    result = subprocess.run([sys.executable, '-c', code], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} import failed:\n{result.stderr}')
    print(f'{label}:', result.stdout.strip().splitlines()[-1])

print('\nToolchain:')
for tool in ['gcc', 'g++', 'aarch64-linux-gnu-gcc', 'arm-linux-gnueabihf-gcc', 'emcc', 'cmake', 'ninja']:
    print(f'{tool}:', shutil.which(tool))


## Pick A Target Profile

A TVM target describes the machine code TVM should generate. The same model can be compiled for local Linux, Raspberry Pi, WebAssembly, Vulkan, or C source export by changing the profile.

For this notebook, use `x86_64` because it runs inside the devcontainer. Cross targets are useful after local validation works.

In [ ]:
for name, profile in sorted(TARGETS.items()):
    print(f'{name}')
    print(f'  target: {profile["target"]}')
    if 'host' in profile:
        print(f'  host:   {profile["host"]}')
    print(f'  output: model.{profile["ext"]}')
    if 'cc' in profile:
        print(f'  cc:     {profile["cc"]}')
    print()

TARGET_PROFILE = 'x86_64'


## Compile An ImageNet Model

This notebook uses torchvision `resnet18` with pretrained ImageNet weights so the runtime step can classify the sample cat image with real labels.

The first run may download the pretrained weights into the Torch cache. After the weights are available, the rest of the flow is local: trace the PyTorch model, import it into Relay, compile with TVM, and run the exported artifacts.


In [ ]:
from compile import import_pytorch

MODEL_NAME = 'resnet18'
INPUT_NAME = 'input0'
INPUT_SHAPE = (1, 3, 224, 224)

mod, params, labels = import_pytorch(MODEL_NAME, INPUT_NAME, INPUT_SHAPE)

print('Model       :', MODEL_NAME)
print('Input       :', INPUT_NAME, INPUT_SHAPE)
print('Relay module:', type(mod).__name__)
print('Parameters  :', len(params))
print('Labels      :', len(labels))


## Import The Model Into Relay

`import_pytorch` in `compilation/compile.py` keeps this step explicit: it loads the torchvision model, traces it to TorchScript with an example tensor, and calls `relay.frontend.from_pytorch` with the input name and shape.

At this stage TVM has understood the model graph but has not generated target-specific machine code yet.


In [ ]:
print('First labels:', ', '.join(labels[:5]))
print('Cat labels:', ', '.join(label for label in labels if 'cat' in label.lower())[:240])


## Compile And Export Artifacts

`build_and_save` from `compilation/compile.py` compiles the Relay module with `relay.build` and writes a portable artifact directory:

- `model.json`: graph-executor graph
- `model.params`: serialized parameter dictionary
- `model.so`, `model.tar`, or `model.wasm`: compiled target library
- `metadata.json`: model name, input name, input shape, target, and library filename
- `labels.txt`: ImageNet labels exported from the torchvision weights

Generated outputs go under `examples/artifacts/`, which is ignored by git except for `.gitkeep`.


In [ ]:
import json
from compile import build_and_save

artifact_dir = build_and_save(
    mod,
    params,
    model_name=MODEL_NAME,
    target_name=TARGET_PROFILE,
    output_dir=ROOT / 'examples' / 'artifacts',
    labels=labels,
    input_name=INPUT_NAME,
    input_shape=INPUT_SHAPE,
)
print('Artifacts:', artifact_dir.relative_to(ROOT))
for path in sorted(artifact_dir.iterdir()):
    print('-', path.name)

metadata = json.loads((artifact_dir / 'metadata.json').read_text())
metadata


## Run The Compiled Model On `cat.png`

The Python runtime path mirrors what a deployment program does:

1. Read `metadata.json` to learn the input name, input shape, compiled library file, and label file.
2. Load `model.json`, `model.params`, and the compiled library.
3. Preprocess `examples/assets/cat.png` into the model's `NCHW` ImageNet input tensor.
4. Create a TVM graph executor module, run inference, and map the top outputs back to labels.


In [ ]:
from tvm_prep.preprocess import load_imagenet_image
from tvm_prep.runtime import run_graph_executor, topk

image_path = ROOT / 'examples' / 'assets' / 'cat.png'
labels_path = artifact_dir / metadata['labels']
compiled_labels = labels_path.read_text(encoding='utf-8').splitlines()

image = load_imagenet_image(image_path, metadata['input_shape'], layout='NCHW')
output = run_graph_executor(artifact_dir, image)
predictions = topk(output, compiled_labels, k=5)

print('Image:', image_path.relative_to(ROOT))
print('Output shape:', output.shape)
print()
for rank, (idx, prob, label) in enumerate(predictions, start=1):
    print(f'{rank}: id={idx} p={prob:.6f} {label}')

top_id, top_prob, top_label = predictions[0]
print(f'\nPrediction: {top_label} ({top_prob:.2%})')


## Compile Real Models With The CLI

The cells above use the import and build functions directly so each TVM step is visible. The same workflow is available from the CLI in `compilation/compile.py`.

PyTorch torchvision example:

```bash
python compilation/compile.py \
  --frontend pytorch \
  --model resnet18 \
  --target x86_64

python examples/python/run_model.py \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image examples/assets/cat.png
```

Supported starter PyTorch models are `resnet18`, `mobilenet_v2`, `squeezenet1_1`, and `shufflenet_v2_x1_0`. TensorFlow starter models are `mobilenet_v2` and `resnet50`.

ONNX and TFLite need a file path, input name, and input shape because those details are model-specific:

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path path/to/model.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --target x86_64
```

## Compile For Other Targets

Change only `--target` when moving from local validation to another supported profile:

```bash
python compilation/compile.py --frontend pytorch --model resnet18 --target raspi4_aarch64
python compilation/compile.py --frontend pytorch --model resnet18 --target raspi_armv7
python compilation/compile.py --frontend pytorch --model resnet18 --target wasm32
python compilation/compile.py --frontend pytorch --model resnet18 --target c
```

Cross-compilation is only half the job. The target device also needs a compatible TVM runtime package. Build it with:

```bash
bash compilation/build_runtime.sh raspi4_aarch64
# Output: examples/artifacts/runtime/raspi4_aarch64/
```

Copy that whole runtime directory alongside the compiled model artifacts to the target device. The package includes `libtvm_runtime.so`, C++ headers, and TVM Python files used by the examples.

## Use The C++ Runtime Example

The C++ runner consumes the same artifact directory as the Python runtime. Build it after compiling a model:

```bash
cmake -S examples/cpp -B examples/cpp/build -G Ninja
cmake --build examples/cpp/build

examples/cpp/build/run_tvm_graph \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image examples/assets/cat.png
```

For cross-compiled artifacts, the C++ runner, compiled model library, packaged TVM runtime, and target OS ABI must all match.

## Where To Change Things

- Add target profiles in `compilation/targets.py`.
- Change compile/export behavior in `compilation/compile.py`.
- Change Python runtime behavior in `examples/python/tvm_prep/runtime.py`.
- Change image preprocessing in `examples/python/tvm_prep/preprocess.py`.

Keep generated model outputs under `examples/artifacts/`. Do not commit generated `.so`, `.params`, `.json`, ONNX, or artifact directories.
